# Data Types in Finance
### Applied Statistical Data Analysis — Prof. Dr. Kristyna Ters | MSc Finance | FHNW
**Based on:** Brooks, C. — *Introductory Econometrics for Finance*, Cambridge University Press

---
**Learning Objectives:**
- Distinguish time series, cross-sectional, and panel data
- Calculate simple and log returns
- Visualise and explore real financial data
- Interpret descriptive statistics in a finance context

> **How to use this notebook:**
> - Run each cell in order with **Shift+Enter**
> - Exercises are marked with 🏋️ — try to solve them before looking at the hints
> - Hints are hidden in collapsed cells — click ▶ to reveal
> - Challenge exercises (🔥) go beyond the lecture material


## Step 0 — Install & Import Libraries

In [ ]:
!pip install yfinance --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11
})
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'; GREY = '#4B4B4B'
print('✓ Libraries loaded. Ready to go!')

---
# Part 1 — Time Series Data

**Definition:** Observations on the *same entity* recorded at *multiple points in time* — in sequential order.

Key property: **order matters**. Yesterday's price directly affects today's return.

### 1.1 Download Stock Price Data

In [ ]:
TICKER = 'AAPL'
START  = '2019-01-01'
END    = '2024-12-31'

data   = yf.download(TICKER, start=START, end=END, auto_adjust=True, progress=False)
prices = data['Close'].squeeze()

print(f'Downloaded {TICKER}: {len(prices)} trading days')
print(f'Date range: {prices.index[0].date()} → {prices.index[-1].date()}')
print(f'\nFirst 5 observations:')
print(prices.head().round(2))
print(f'\nLast 5 observations:')
print(prices.tail().round(2))

### 1.2 Plot Price Series

In [ ]:
ret = prices.pct_change().dropna() * 100

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                          gridspec_kw={'height_ratios': [2, 1], 'hspace': 0.05})
axes[0].plot(prices.index, prices.values, color='black', lw=1.3)
axes[0].fill_between(prices.index, prices.values, alpha=0.06, color='black')
axes[0].set_ylabel('Price (USD)')
axes[0].set_title(f'{TICKER} — Price & Daily Returns ({START[:4]}–{END[:4]})', fontweight='bold')

axes[1].bar(ret.index, ret.values,
            color=[RED if v < 0 else 'black' for v in ret.values], width=1.0)
axes[1].axhline(0, color=GREY, lw=0.8)
axes[1].set_ylabel('Daily return (%)')
axes[1].set_xlabel('Date')
plt.tight_layout()
plt.show()

---
## 🏋️ Exercise 1.1 — Identify the Data Type

For each dataset below, state whether it is **time series**, **cross-sectional**, or **panel** data. Justify your answer in one sentence.

| # | Dataset | Data Type | Justification |
|---|---------|-----------|---------------|
| a | Daily EUR/USD exchange rate, Jan 2015 – Dec 2024 | ? | ? |
| b | P/E ratios of 50 S&P 500 firms on 31 December 2023 | ? | ? |
| c | Quarterly revenue of Apple, Microsoft, and Google, 2018–2024 | ? | ? |
| d | Beta of Nestlé estimated using monthly returns 2010–2020 | ? | ? |
| e | Credit ratings of 200 European banks in Q1 2024 | ? | ? |
| f | Monthly returns of the SMI index, 2000–2024 | ? | ? |

**Your answers:** (double-click this cell to edit)

| # | Data Type | Justification |
|---|-----------|---------------|
| a | | |
| b | | |
| c | | |
| d | | |
| e | | |
| f | | |

<details>
<summary>💡 Click here for answers</summary>

| # | Data Type | Justification |
|---|-----------|---------------|
| a | Time series | One entity (EUR/USD), many time points, order matters |
| b | Cross-sectional | Many entities (firms), one point in time |
| c | Panel | Multiple entities (firms) observed over multiple periods |
| d | Time series | One entity (Nestlé), data ordered over time |
| e | Cross-sectional | Many entities (banks), single snapshot in time |
| f | Time series | One entity (SMI), ordered over time |

</details>

---
## 🏋️ Exercise 1.2 — Explore a Different Stock

Change the ticker to a **Swiss stock** (e.g. `NESN.SW` for Nestlé, `NOVN.SW` for Novartis, `ROG.SW` for Roche) and re-run the cells above.

Answer these questions after running:

In [ ]:
# Change the ticker here and re-run
TICKER_EX = 'NESN.SW'
START_EX  = '2019-01-01'
END_EX    = '2024-12-31'

data_ex   = yf.download(TICKER_EX, start=START_EX, end=END_EX, auto_adjust=True, progress=False)
prices_ex = data_ex['Close'].squeeze()
ret_ex    = prices_ex.pct_change().dropna() * 100

print(f'Stock: {TICKER_EX}')
print(f'Number of trading days: {len(prices_ex)}')
print(f'Price range: {prices_ex.min():.2f} – {prices_ex.max():.2f}')
print(f'\nReturn statistics (%):')
print(ret_ex.describe().round(4))
print(f'\nWorst single day: {ret_ex.min():.2f}% on {ret_ex.idxmin().date()}')
print(f'Best single day:  {ret_ex.max():.2f}% on {ret_ex.idxmax().date()}')

**Questions** (edit this cell):
1. On what date did the stock have its worst single day, and what might have caused it?
2. How does the standard deviation compare to AAPL's (~1.5% daily)? Is this stock more or less volatile?
3. Was the mean daily return positive? What does this mean economically?

**Your answers:**
1. 
2. 
3. 

---
# Part 2 — Simple vs. Log Returns

| | Simple Return | Log Return |
|--|--|--|
| **Formula** | $R_t = (P_t - P_{t-1})/P_{t-1}$ | $r_t = \ln(P_t/P_{t-1})$ |
| **Python** | `prices.pct_change()` | `np.log(prices/prices.shift(1))` |
| **Additive?** | No — must multiply | Yes — just sum |
| **Use for** | Reporting, portfolio returns | Modelling, statistics, regressions |

### 2.1 Calculate Both

In [ ]:
simple_ret = prices.pct_change().dropna()
log_ret    = np.log(prices / prices.shift(1)).dropna()

idx = simple_ret.index.intersection(log_ret.index)
s   = simple_ret.loc[idx] * 100
l   = log_ret.loc[idx]    * 100

print('Descriptive Statistics (%)')
print(pd.DataFrame({'Simple': s.describe(), 'Log': l.describe()}).round(5))
print(f'\nMax daily difference |simple − log|: {(s-l).abs().max():.5f}%')

---
## 🏋️ Exercise 2.1 — Manual Calculation

Compute simple and log returns **by hand** for these price sequences. Then verify with Python.

In [ ]:
# ── Scenario A: Small move ──────────────────────────────────────
P0_a, P1_a = 100, 101

# YOUR CODE HERE — calculate simple and log return for scenario A
# simple_a = ...
# log_a    = ...

# ── Scenario B: Large positive move ─────────────────────────────
P0_b, P1_b = 100, 150

# YOUR CODE HERE
# simple_b = ...
# log_b    = ...

# ── Scenario C: Large negative move (crash) ─────────────────────
P0_c, P1_c = 100, 65

# YOUR CODE HERE
# simple_c = ...
# log_c    = ...

# ── Scenario D: Two-day round trip: 100 → 80 → 100 ─────────────
P0_d, P1_d, P2_d = 100, 80, 100

# YOUR CODE HERE — show that log returns add up to 0, simple do not
# simple_d1 = ...; simple_d2 = ...; total_simple = (1+simple_d1)*(1+simple_d2) - 1
# log_d1    = ...; log_d2    = ...; total_log    = log_d1 + log_d2

print('Fill in your code above, then run!')

<details>
<summary>💡 Solution</summary>

```python
# Scenario A
simple_a = (P1_a - P0_a) / P0_a        # = 0.01 = 1%
log_a    = np.log(P1_a / P0_a)         # = 0.00995 ≈ 1% (nearly equal for small moves)

# Scenario B
simple_b = (P1_b - P0_b) / P0_b        # = 0.50 = 50%
log_b    = np.log(P1_b / P0_b)         # = 0.405 = 40.5% (log < simple for large moves)

# Scenario C
simple_c = (P1_c - P0_c) / P0_c        # = -0.35 = -35%
log_c    = np.log(P1_c / P0_c)         # = -0.431 = -43.1% (log more negative)

# Scenario D: round trip
simple_d1 = (P1_d - P0_d) / P0_d      # = -20%
simple_d2 = (P2_d - P1_d) / P1_d      # = +25%
total_simple = (1+simple_d1)*(1+simple_d2) - 1  # = -0.20*1.25 - 1 = 0 ✓ (back to 100)
# But you must MULTIPLY simple returns — not intuitive!

log_d1 = np.log(P1_d / P0_d)          # = ln(0.8) = -0.2231
log_d2 = np.log(P2_d / P1_d)          # = ln(1.25) = +0.2231
total_log = log_d1 + log_d2            # = 0.000 ← just add! ✓
```
**Key insight:** Log returns for a round trip add up to exactly 0 — this is why they're preferred in multi-period models.
</details>

---
## 🏋️ Exercise 2.2 — Annualise Returns and Volatility

Using the AAPL data from Part 1, compute:
- Annualised mean return
- Annualised volatility
- Approximate Sharpe Ratio (assume risk-free rate = 4%)

In [ ]:
TRADING_DAYS = 252
RISK_FREE    = 0.04  # 4% annual

daily_mean = simple_ret.mean()
daily_std  = simple_ret.std()

# YOUR CODE HERE
# ann_return = ...
# ann_vol    = ...
# sharpe     = ...

# print(f'Annualised return: {ann_return*100:.2f}%')
# print(f'Annualised vol:    {ann_vol*100:.2f}%')
# print(f'Sharpe ratio:      {sharpe:.2f}')

print('Fill in your code above!')

<details>
<summary>💡 Solution</summary>

```python
ann_return = (1 + daily_mean) ** TRADING_DAYS - 1
ann_vol    = daily_std * np.sqrt(TRADING_DAYS)
sharpe     = (ann_return - RISK_FREE) / ann_vol

print(f'Annualised return: {ann_return*100:.2f}%')
print(f'Annualised vol:    {ann_vol*100:.2f}%')
print(f'Sharpe ratio:      {sharpe:.2f}')
```
**Interpretation:** A Sharpe > 1.0 is generally considered good. AAPL typically shows a Sharpe around 1.0–1.5 over this period.
</details>

---
## 🏋️ Exercise 2.3 — COVID Crash: Returns in Perspective

Zoom into the **COVID crash period** (February–April 2020) and analyse what happened.

In [ ]:
# Filter for COVID period
covid_start = '2020-01-01'
covid_end   = '2020-06-30'

prices_covid = prices.loc[covid_start:covid_end]
ret_covid    = simple_ret.loc[covid_start:covid_end] * 100

# YOUR CODE HERE:
# 1. Find the date of the single worst day
# worst_day  = ...
# worst_ret  = ...

# 2. Calculate the peak-to-trough drawdown
# peak      = prices_covid.max()
# trough    = prices_covid.min()
# drawdown  = (trough - peak) / peak * 100

# 3. How many days from peak to trough?
# peak_date   = ...
# trough_date = ...
# days_down   = ...

# 4. Plot the crash period
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(prices_covid.index, prices_covid.values, color='black', lw=1.5)
ax.set_title('AAPL — COVID Crash Period (Jan–Jun 2020)', fontweight='bold')
ax.set_ylabel('Price (USD)')
plt.tight_layout()
plt.show()

print('Add your calculations above!')

<details>
<summary>💡 Solution</summary>

```python
worst_day  = ret_covid.idxmin()
worst_ret  = ret_covid.min()
print(f'Worst single day: {worst_ret:.2f}% on {worst_day.date()}')

peak        = prices_covid.max()
trough      = prices_covid.min()
drawdown    = (trough - peak) / peak * 100
print(f'Peak-to-trough drawdown: {drawdown:.1f}%')

peak_date   = prices_covid.idxmax()
trough_date = prices_covid.idxmin()
days_down   = (trough_date - peak_date).days
print(f'Days from peak to trough: {days_down} calendar days')
```
</details>

---
# Part 3 — Cross-Sectional Data & Beta

**Definition:** Observations on *multiple entities* at a *single point in time*.

**Finance application:** Estimating **market beta** (β) — the sensitivity of a stock to market movements.

$$\beta = \frac{\text{Cov}(r_i, r_m)}{\text{Var}(r_m)}$$

### 3.1 Download Multiple Stocks

In [ ]:
tickers = ['AAPL', 'MSFT', 'JPM', '^GSPC']
raw     = yf.download(tickers, start='2019-01-01', end='2024-12-31',
                      auto_adjust=True, progress=False)
prices_panel = raw['Close'].copy()
prices_panel.columns = ['AAPL', 'MSFT', 'JPM', 'SP500']
prices_panel = prices_panel.dropna()
ret_panel    = prices_panel.pct_change().dropna()

print(f'Panel shape: {ret_panel.shape}  ({ret_panel.shape[0]} days × {ret_panel.shape[1]} assets)')
print(f'\nSummary statistics (daily returns, %):')
print((ret_panel * 100).describe().round(4))

### 3.2 Estimate Beta

In [ ]:
def estimate_beta(stock_ret, market_ret):
    """Estimate beta using OLS regression."""
    slope, intercept, r, p, se = stats.linregress(market_ret, stock_ret)
    return {'Beta': round(slope, 3),
            'Alpha (daily %)': round(intercept * 100, 4),
            'R²': round(r**2, 3),
            'p-value (beta)': round(p, 4)}

results = {}
for col in ['AAPL', 'MSFT', 'JPM']:
    results[col] = estimate_beta(ret_panel[col], ret_panel['SP500'])

beta_df = pd.DataFrame(results).T
print('Cross-Sectional Beta Estimates (full period 2019–2024):')
print(beta_df)
print('\n→ CROSS-SECTIONAL: one row per firm, estimated at one point in time')

---
## 🏋️ Exercise 3.1 — Interpret Beta

Using the beta estimates above, answer the following:

**Questions** (edit this cell):

1. If the S&P 500 drops by **2%** in one day, by approximately how much would you expect each stock to drop?
   - AAPL: _%
   - MSFT: _%
   - JPM: _%

2. Which stock is the most **defensive** (least sensitive to market movements)? Why might that be?

3. The R² for AAPL tells us that ___% of AAPL's daily return variation is explained by the market. What explains the rest?

4. What does a **positive alpha** mean economically? Is it sustainable long-term?

**Your answers:**
1. AAPL: , MSFT: , JPM:
2. 
3. 
4. 

<details>
<summary>💡 Answers</summary>

1. Multiply beta × market move. E.g. if AAPL β ≈ 1.2: expected drop = 1.2 × 2% = **2.4%**
2. The stock with β closest to 0 is most defensive — typically JPM or MSFT. Financial sector sensitivity depends on interest rates, not just market moves.
3. R² = 0.63 → 63% explained by market. The remaining 37% is **idiosyncratic risk** (company-specific news, earnings surprises, product launches).
4. Positive alpha = stock outperforms the market on risk-adjusted basis. Hard to sustain over long periods — most alphas disappear as markets become more efficient.
</details>

---
## 🏋️ Exercise 3.2 — Add Your Own Stock

Estimate beta for a stock of your choice and compare it to the market.

In [ ]:
# Choose any stock — try 'TSLA', 'NVDA', 'NESN.SW', 'NOVN.SW', 'UBS', 'CS'
MY_STOCK = 'TSLA'

# Download
my_data   = yf.download(MY_STOCK, start='2019-01-01', end='2024-12-31',
                         auto_adjust=True, progress=False)
my_prices = my_data['Close'].squeeze()
my_ret    = my_prices.pct_change().dropna()

# Align with S&P 500
common_idx = my_ret.index.intersection(ret_panel['SP500'].index)
my_ret_aligned = my_ret.loc[common_idx]
sp_ret_aligned = ret_panel['SP500'].loc[common_idx]

# YOUR CODE: estimate beta
# my_beta = estimate_beta(my_ret_aligned, sp_ret_aligned)
# print(f'Beta estimates for {MY_STOCK}:')
# print(my_beta)

# YOUR CODE: scatter plot
# fig, ax = plt.subplots(figsize=(7, 5))
# ax.scatter(sp_ret_aligned * 100, my_ret_aligned * 100, alpha=0.2, s=8, color='black')
# ... add regression line ...
# ax.set_xlabel('S&P 500 daily return (%)')
# ax.set_ylabel(f'{MY_STOCK} daily return (%)')
# ax.set_title(f'Beta regression: {MY_STOCK} vs S&P 500')
# plt.show()

print(f'Choose your stock and fill in the code above!')

<details>
<summary>💡 Solution</summary>

```python
my_beta = estimate_beta(my_ret_aligned, sp_ret_aligned)
print(f'Beta estimates for {MY_STOCK}:')
for k, v in my_beta.items():
    print(f'  {k}: {v}')

# Regression line
b = my_beta['Beta']
a = my_beta['Alpha (daily %)'] / 100
x_line = np.linspace(sp_ret_aligned.min(), sp_ret_aligned.max(), 100)
y_line = a + b * x_line

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(sp_ret_aligned * 100, my_ret_aligned * 100, alpha=0.2, s=8, color='black')
ax.plot(x_line * 100, y_line * 100, color=RED, lw=2,
        label=f'β = {b:.2f}, R² = {my_beta["R²"]:.2f}')
ax.axhline(0, color=GREY, lw=0.7)
ax.axvline(0, color=GREY, lw=0.7)
ax.set_xlabel('S&P 500 daily return (%)')
ax.set_ylabel(f'{MY_STOCK} daily return (%)')
ax.set_title(f'Beta regression: {MY_STOCK} vs S&P 500', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()
```
</details>

---
## 🏋️ Exercise 3.3 — Rolling Beta

Beta is not constant over time! Compute a **rolling 6-month beta** for AAPL and plot how it changes.

In [ ]:
# Rolling beta — window of 126 trading days ≈ 6 months
WINDOW = 126

rolling_betas = []
rolling_dates = []

for i in range(WINDOW, len(ret_panel)):
    window_data = ret_panel.iloc[i-WINDOW:i]
    b = estimate_beta(window_data['AAPL'], window_data['SP500'])['Beta']
    rolling_betas.append(b)
    rolling_dates.append(ret_panel.index[i])

rolling_beta_series = pd.Series(rolling_betas, index=rolling_dates)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(rolling_beta_series.index, rolling_beta_series.values, color='black', lw=1.5)
ax.axhline(1.0, color=RED, lw=1.2, ls='--', label='β = 1 (market)')
ax.axhline(rolling_beta_series.mean(), color=ORANGE, lw=1.2, ls=':',
           label=f'Mean β = {rolling_beta_series.mean():.2f}')
ax.fill_between(rolling_beta_series.index, rolling_beta_series.values, 1.0,
                where=rolling_beta_series.values > 1, alpha=0.15, color=RED)
ax.set_title('AAPL Rolling 6-Month Beta vs. S&P 500', fontweight='bold')
ax.set_ylabel('Beta (β)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Beta range: {rolling_beta_series.min():.2f} – {rolling_beta_series.max():.2f}')
print(f'\nQuestion: When was AAPL most sensitive to the market?')
print(f'Date of max beta: {rolling_beta_series.idxmax().date()} (β = {rolling_beta_series.max():.2f})')

**Questions** (edit this cell):
1. Does beta appear stable over time, or does it fluctuate? What are the implications for risk management?
2. Was AAPL more or less sensitive to the market during the COVID crash (early 2020)?
3. What would it mean if beta temporarily went **below 1** for AAPL?

**Your answers:**
1. 
2. 
3. 

---
# Part 4 — Panel Data

**Definition:** Multiple entities observed over multiple time periods.

### 4.1 Cumulative Returns & Heatmap

In [ ]:
try:
    import seaborn as sns
    HAS_SNS = True
except ImportError:
    !pip install seaborn --quiet
    import seaborn as sns
    HAS_SNS = True

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Cumulative returns
cum = (1 + ret_panel).cumprod() - 1
for col, c in zip(cum.columns, ['black', ORANGE, '#0E75FE', RED]):
    axes[0].plot(cum.index, cum[col] * 100, lw=1.6, color=c, label=col)
axes[0].axhline(0, color=GREY, lw=0.8)
axes[0].set_title('Cumulative Returns 2019–2024', fontweight='bold')
axes[0].set_ylabel('Cumulative return (%)')
axes[0].legend()

# Monthly heatmap
monthly = ret_panel.resample('ME').apply(lambda x: (1+x).prod()-1)
mp = monthly.iloc[-24:] * 100
mp.index = mp.index.strftime('%b %y')
sns.heatmap(mp.T, ax=axes[1], cmap='RdYlGn', center=0,
            annot=True, fmt='.0f', linewidths=0.5, linecolor='white',
            annot_kws={'size': 8}, cbar_kws={'label': 'Monthly return (%)'})
axes[1].set_title('Monthly Returns — Last 24 Months', fontweight='bold')
plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

---
## 🏋️ Exercise 4.1 — Correlation Analysis

Compute and visualise the **correlation matrix** between the four assets.

In [ ]:
# YOUR CODE HERE:
# 1. Compute the correlation matrix of ret_panel
# corr = ...
# print(corr.round(3))

# 2. Plot as heatmap
# fig, ax = plt.subplots(figsize=(6, 5))
# sns.heatmap(corr, ax=ax, annot=True, fmt='.2f', cmap='coolwarm',
#             center=0, vmin=-1, vmax=1, linewidths=0.5)
# ax.set_title('Correlation Matrix — Daily Returns', fontweight='bold')
# plt.tight_layout()
# plt.show()

print('Fill in your code!')

<details>
<summary>💡 Solution</summary>

```python
corr = ret_panel.corr()
print('Correlation Matrix:')
print(corr.round(3))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, ax=ax, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            annot_kws={'size': 13, 'fontweight': 'bold'})
ax.set_title('Correlation Matrix — Daily Returns (2019–2024)', fontweight='bold')
plt.tight_layout()
plt.show()
```
</details>

**Questions** (edit this cell):
1. Which two stocks are most correlated? Why does this make sense economically?
2. Does the S&P 500 have the highest or lowest correlation with individual stocks? Why?
3. From a portfolio perspective, which stock provides the best **diversification benefit** when added to a position in AAPL?

**Your answers:**
1. 
2. 
3. 

---
## 🏋️ Exercise 4.2 — Crisis vs. Normal Times

A key stylized fact: **correlations rise during crises**. Test this for the COVID crash.

In [ ]:
# Split into crisis and normal periods
crisis = ret_panel.loc['2020-02-01':'2020-05-31']
normal = ret_panel.loc['2021-01-01':'2021-12-31']

# YOUR CODE HERE:
# corr_crisis = ...
# corr_normal = ...

# fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# sns.heatmap(corr_crisis, ax=axes[0], annot=True, fmt='.2f', cmap='coolwarm',
#             center=0, vmin=-1, vmax=1)
# axes[0].set_title('COVID Crisis (Feb–May 2020)', fontweight='bold')
# sns.heatmap(corr_normal, ax=axes[1], annot=True, fmt='.2f', cmap='coolwarm',
#             center=0, vmin=-1, vmax=1)
# axes[1].set_title('Normal Period (2021)', fontweight='bold')
# plt.tight_layout()
# plt.show()

print('Fill in your code!')

<details>
<summary>💡 Solution + Interpretation</summary>

```python
corr_crisis = crisis.corr()
corr_normal = normal.corr()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(corr_crisis, ax=axes[0], annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, annot_kws={'size': 13})
axes[0].set_title('COVID Crisis (Feb–May 2020)', fontweight='bold')
sns.heatmap(corr_normal, ax=axes[1], annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, annot_kws={'size': 13})
axes[1].set_title('Normal Period (2021)', fontweight='bold')
plt.tight_layout()
plt.show()
```

**Expected finding:** Correlations between stocks are significantly higher during COVID than in normal times. This is the **correlation breakdown** phenomenon — diversification fails exactly when you need it most, because all stocks fall together in a panic.
</details>

---
# Part 5 — Annualised Statistics

In [ ]:
TRADING_DAYS = 252
RF = 0.04  # risk-free rate

print(f'Annualised Statistics (2019–2024) | Risk-free rate: {RF*100:.0f}%')
print('=' * 72)
for col in ret_panel.columns:
    r    = ret_panel[col]
    ann_ret = (1 + r.mean()) ** TRADING_DAYS - 1
    ann_vol = r.std() * np.sqrt(TRADING_DAYS)
    sharpe  = (ann_ret - RF) / ann_vol
    print(f'{col:<8} | Return: {ann_ret*100:6.1f}%  '
          f'| Vol: {ann_vol*100:5.1f}%  '
          f'| Sharpe: {sharpe:5.2f}  '
          f'| Max DD: {((ret_panel[col]+1).cumprod().div((ret_panel[col]+1).cumprod().cummax())-1).min()*100:.1f}%')

---
## 🏋️ Exercise 5.1 — Build a Simple Portfolio

Construct an **equally-weighted portfolio** (25% each) and compare it to the S&P 500.

In [ ]:
# Equally-weighted portfolio (excluding SP500 from portfolio)
stocks = ['AAPL', 'MSFT', 'JPM']
weights = np.array([1/3, 1/3, 1/3])

# YOUR CODE HERE:
# 1. Calculate portfolio daily returns
# port_ret = (ret_panel[stocks] * weights).sum(axis=1)

# 2. Calculate cumulative returns for portfolio vs S&P 500
# cum_port = (1 + port_ret).cumprod() - 1
# cum_sp   = (1 + ret_panel['SP500']).cumprod() - 1

# 3. Plot
# fig, ax = plt.subplots(figsize=(13, 5))
# ax.plot(cum_port.index, cum_port*100, color='black', lw=2, label='EW Portfolio (AAPL+MSFT+JPM)')
# ax.plot(cum_sp.index,   cum_sp*100,   color=RED, lw=2, ls='--', label='S&P 500')
# ...

# 4. Compare Sharpe ratios

print('Fill in your code!')

<details>
<summary>💡 Solution</summary>

```python
port_ret = (ret_panel[stocks] * weights).sum(axis=1)
cum_port = (1 + port_ret).cumprod() - 1
cum_sp   = (1 + ret_panel['SP500']).cumprod() - 1

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(cum_port.index, cum_port*100, color='black', lw=2,
        label='EW Portfolio (AAPL+MSFT+JPM)')
ax.plot(cum_sp.index, cum_sp*100, color=RED, lw=2, ls='--', label='S&P 500')
ax.axhline(0, color=GREY, lw=0.8)
ax.set_title('Equally-Weighted Portfolio vs. S&P 500', fontweight='bold')
ax.set_ylabel('Cumulative return (%)')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Sharpe comparison
for label, r in [('EW Portfolio', port_ret), ('S&P 500', ret_panel['SP500'])]:
    ann_r = (1+r.mean())**252 - 1
    ann_v = r.std() * np.sqrt(252)
    print(f'{label}: Return={ann_r*100:.1f}%, Vol={ann_v*100:.1f}%, Sharpe={(ann_r-0.04)/ann_v:.2f}')
```
</details>

---
# 🔥 Challenge Exercises

These go beyond the lecture material — try them if you want to push further.

---
## 🔥 Challenge 1 — Sub-period Beta Stability

Split the sample into three periods (pre-COVID, COVID, post-COVID) and estimate beta separately for each. How much does beta change across periods?

In [ ]:
periods = {
    'Pre-COVID (2019)':       ('2019-01-01', '2020-01-31'),
    'COVID crisis (2020)':    ('2020-02-01', '2020-12-31'),
    'Post-COVID (2021–2024)': ('2021-01-01', '2024-12-31'),
}

# YOUR CODE: estimate beta for AAPL in each sub-period and print a comparison table
print('Estimate beta for each period!')

---
## 🔥 Challenge 2 — Momentum Strategy

A simple momentum strategy: each month, invest in the stock with the **highest return last month**. Compare to buy-and-hold S&P 500.

In [ ]:
# Monthly returns for AAPL, MSFT, JPM
monthly_ret = ret_panel[['AAPL','MSFT','JPM']].resample('ME').apply(
    lambda x: (1+x).prod()-1
)

# YOUR CODE:
# Each month, pick the stock with highest return last month
# Then track: what did that stock return THIS month?
# Compare cumulative momentum strategy vs. S&P 500

print('Build your momentum strategy!')

---
## 🔥 Challenge 3 — Visualise Risk-Return Trade-off

Create a **risk-return scatter plot** (annualised vol on x-axis, annualised return on y-axis) for at least 10 stocks.

In [ ]:
universe = ['AAPL','MSFT','JPM','TSLA','NVDA','JNJ','XOM','AMZN','META','GS','^GSPC']

# YOUR CODE:
# 1. Download all tickers
# 2. Compute annualised return and vol for each
# 3. Scatter plot: x = vol, y = return, label each point with ticker
# 4. Draw the S&P 500 as a reference point

print('Build your risk-return chart!')

---
## Final Summary

| Concept | Key Formula | Python |
|---------|-------------|--------|
| Simple return | $R_t = (P_t-P_{t-1})/P_{t-1}$ | `prices.pct_change()` |
| Log return | $r_t = \ln(P_t/P_{t-1})$ | `np.log(prices/prices.shift(1))` |
| Annual vol | $\sigma_{ann} = \sigma_{daily} \times \sqrt{252}$ | `ret.std() * np.sqrt(252)` |
| Beta | $\beta = \text{Cov}(r_i, r_m)/\text{Var}(r_m)$ | `stats.linregress(market, stock)` |
| Sharpe | $(r - r_f) / \sigma$ | manual calculation |

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*

*Next topic: Statistical Properties of Financial Data (Fat Tails, Skewness, Volatility Clustering)*